# AquaHealth AI — real-data training on Colab (GPU)

Reproducible setup for training the EfficientNet-B0 classifier on the frozen split.
Everything substantive runs repository code (`scripts/`, `src/`); this notebook only orchestrates.

**Before running — manual steps (Colab cannot see your laptop):**
1. `Runtime → Change runtime type → GPU` (T4 is enough; A100/L4 if available).
2. Put the dataset in your Google Drive at `MyDrive/AquaHealth/`:
   either the extracted folder `New Dataset/` (with `train_split/`, `test_split/`, `test.csv`)
   or the original `DATASET.zip` (preferred: unzipping to the VM is much faster than reading 3,500 files from Drive).
   The notebook never modifies Drive contents; it only reads them.
3. Run the cells top to bottom. The last cell is the full training command and is **disabled by default**.

Repository: `https://github.com/kolursamith/aquahealth.git` — branch `feature/real-data-training`.

## 1. Runtime, GPU and versions

In [ ]:
!nvidia-smi
import sys, platform
print("Python", platform.python_version())

In [ ]:
import os, subprocess, pathlib
REPO_DIR = pathlib.Path("/content/aquahealth")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "feature/real-data-training", "--single-branch", "https://github.com/kolursamith/aquahealth.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

### Dependencies
Installs the project's pinned runtime requirements (`requirements/base.txt`) plus pytest.
Colab ships its own torch; the pin in `base.txt` is what the code was validated against, so it is installed explicitly
(this can download ~2 GB of CUDA wheels). Versions before and after are printed so nothing is upgraded silently.

In [ ]:
import importlib.metadata as md
def versions():
    out = {}
    for name in ("torch", "torchvision", "numpy", "pillow", "opencv-python-headless"):
        try: out[name] = md.version(name)
        except md.PackageNotFoundError: out[name] = None
    return out
print("before:", versions())
!pip install -q -r requirements/base.txt pytest==9.1.1
print("after: ", versions())
!python scripts/verify_environment.py --requirements requirements/base.txt

In [ ]:
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, f"| memory {p.total_memory / 1024**3:.1f} GB")
else:
    raise SystemExit("No GPU: set Runtime -> Change runtime type -> GPU and restart.")

## 2. Dataset: mount Google Drive, reference the data read-only

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/AquaHealth")
assert DRIVE.exists(), f"Create {DRIVE} in your Drive and put 'New Dataset/' or DATASET.zip inside it"
print(sorted(p.name for p in DRIVE.iterdir()))

In [ ]:
# The repo expects data/original/aquahealth -> <dataset root with train_split/, test_split/, test.csv>.
# Prefer unzipping DATASET.zip onto the VM disk (fast, ephemeral); fall back to the extracted Drive folder.
import shutil
LOCAL = pathlib.Path("/content/aquahealth_data")
zip_path = DRIVE / "DATASET.zip"
extracted = DRIVE / "New Dataset"
if zip_path.exists():
    if not (LOCAL / "New Dataset").exists():
        LOCAL.mkdir(exist_ok=True)
        !unzip -q "{zip_path}" -d "{LOCAL}"
    dataset_root = LOCAL / "New Dataset"
elif extracted.exists():
    dataset_root = extracted   # read directly from Drive (slower per epoch)
else:
    raise SystemExit("Neither DATASET.zip nor 'New Dataset/' found in Drive")
link = REPO_DIR / "data" / "original" / "aquahealth"
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(dataset_root)
print("data/original/aquahealth ->", dataset_root)
print(sorted(p.name for p in link.iterdir()))

## 3. Frozen split, class mapping, dataset safety
`tests/test_frozen_manifest.py` verifies the committed manifest's digest, structure, 70/15/15 counts and canonical
labels, and (now that the files are present) that every manifest path exists. `tests/test_manifest.py` covers the
split logic itself. The frozen `test` split is never touched by training.

In [ ]:
!python -m pytest -q tests/test_frozen_manifest.py tests/test_manifest.py

In [ ]:
# Corrupt-file / duplicate re-check against the copy Colab sees (dry run: the manifest is NOT regenerated).
!python scripts/build_split_manifest.py --dry-run --results-dir /content/audit_check | head -20
!cat data/split_manifest.sha256

## 4. Preprocessing, model and one-batch smoke test on the GPU
`scripts/smoke_train.py` checks: train = CLAHE → RandomResizedCrop/flip/rotation/jitter → tensor → ImageNet normalise;
val/test = CLAHE → Resize 256 → CenterCrop 224 → tensor → ImageNet normalise, with **no** augmentation;
EfficientNet-B0 (ImageNet weights) with an 8-way head; then forward → loss → backward → step with finite gradients
on the GPU under mixed precision. Full training must not start unless this prints `SMOKE TEST PASS`.

In [ ]:
import torch, os, time
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
BATCH_SIZE = 32 if gpu_gb < 12 else 64 if gpu_gb < 30 else 128   # conservative; EfficientNet-B0 @224 is light
WORKERS = min(4, os.cpu_count() or 2)
RUN_NAME = time.strftime("%Y%m%d-%H%M%S")
OUTPUT_DIR = DRIVE / "runs" / RUN_NAME          # persistent: survives a runtime disconnect
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"batch size {BATCH_SIZE} (GPU {gpu_gb:.0f} GB) | workers {WORKERS} | outputs -> {OUTPUT_DIR}")

In [ ]:
!python scripts/smoke_train.py --amp --batch-size {BATCH_SIZE} --workers {WORKERS} --json "{OUTPUT_DIR}/smoke_test.json"

## 5. Full training (disabled until the smoke test passes and you flip the switch)
Staged fine-tuning through the project engine: Stage A head-only → Stage B last 3 blocks → Stage C all blocks,
validation every epoch, `best.pt` chosen on validation Macro-F1, checkpoints written to Drive each epoch
(resumable with `--resume <stage>/last.pt`). CLAHE is ON, mixed precision ON, pinned memory ON.
The frozen `test` split is not read by this command.

In [ ]:
RUN_FULL_TRAINING = False   # set True only after "SMOKE TEST PASS" above
cmd = (
    f"python -m src.finetune --manifest --checkpoint-dir '{OUTPUT_DIR}' "
    f"--clahe --amp --pin-memory --batch-size {BATCH_SIZE} --workers {WORKERS} "
    f"--selection-metric f1_macro --seed 42 2>&1 | tee '{OUTPUT_DIR}/train.log'"
)
print(cmd)
if RUN_FULL_TRAINING:
    !{cmd}

## 6. After training — validation report (the `test` split stays untouched until the final, one-shot evaluation)

In [ ]:
# !python -m src.evaluate --checkpoint "{OUTPUT_DIR}/full/best.pt" --split val --out-dir "{OUTPUT_DIR}/eval_val"